# Where did the moisture come from? Vapour transport, back trajectories and the land surface

**27 July 2026 Chicagoland tornado outbreak — Notebook 2 of 3**

Notebook 1 established the mesoscale environment: a warm-sector airmass over northern Illinois with
extreme instability and enough low-level shear to support the seven confirmed tornadoes that touched
down between 11:54 am and 12:59 pm CDT. That instability was built on water vapour. Precipitable
water over the tornado corridor reached **2.29 in (58 mm)**, a value that puts the column near the
top of the July distribution for the site.

This notebook asks the upstream question: **where did that water vapour come from, and what happened
to it on the way?** We answer it three ways — a synoptic moisture-transport diagnosis (integrated
vapour transport), isobaric back trajectories from the storm inflow, and a land-surface budget along
the parcel path. The last of these turns out to be the bridge to Notebook 3, because the wet soil
that helped moisten the inflow is the same wet soil that could not absorb the rain four days later.

## 1. Motivation and the question being asked

Severe-storm forecasting habitually treats moisture as a local quantity: you read the dewpoint, you
compute CAPE, you move on. But a 79 °F dewpoint over Chicago in late July is not made locally. The
boundary layer over northern Illinois cannot evaporate that much water into itself in a day; the
vapour has to be imported. If we want to say anything defensible on air about *why* this event was
as unstable as it was, we need to trace the import.

Three specific questions structure the analysis:

1. **How anomalous was the transport?** Integrated vapour transport (IVT) is the vertically
   integrated horizontal moisture flux, the standard metric for moisture plumes. A number in
   context is worth more than a map.
2. **Where was the air four days earlier?** Back trajectories at several levels, because the answer
   is level-dependent and only some of the column matters for storm inflow.
3. **Did the land surface along the path add moisture or take it away?** A parcel crossing hot dry
   ground mixes with dry air above and loses vapour; a parcel crossing saturated ground with the
   surface energy budget dominated by evaporation gains it.

Each question is answered with a number, and each number is checked against something independent of
the model that produced it.

## 2. Data sources

| Source | Endpoint / access | Use here |
|---|---|---|
| GFS 0.25° analyses | NOMADS / AWS `noaa-gfs-bdp-pds` via [Herbie](https://herbie.readthedocs.io) — `model="gfs"`, `product="pgrb2.0p25"`, `fxx=0` | SPFH / UGRD / VGRD on 17 pressure levels, PWAT, SOILW, LAND |
| GFS 0.25° short-range forecast | same, `fxx=9` from the 12 UTC cycle | time-averaged LHTFL / SHTFL surface fluxes over 18–21 UTC |
| HRRR analysis | `noaa-hrrr-bdp-pds` via Herbie (Notebook 1) | IVT at the Glen Ellyn inflow point at 17 UTC |
| ASOS hourly and daily climate | Iowa Environmental Mesonet, `mesonet.agron.iastate.edu/cgi-bin/request/asos.py` and `.../daily.py` | O'Hare dewpoint record 1990–2026; 30-day rainfall at seven stations near the path, 1995–2025 |

Analysis window for the transport diagnosis: **23 July 00 UTC to 28 July 00 UTC at 6-hourly analysis
steps**, over a domain spanning **15–52 °N, 110–70 °W** — the Gulf of Mexico through the Great Lakes.
Four days of upstream history is enough to carry a boundary-layer parcel from the Gulf coast to
Illinois at typical 925 hPa speeds, and the domain is wide enough that the trajectory does not exit
through a boundary before it gets there.

### Warning: GFS grid and record conventions

These four cost real debugging time. Every one of them fails *silently* — no exception, just a wrong
answer — which is the worst failure mode there is.

- **Field availability.** `SPFH`, `UGRD` and `VGRD` are on 33 pressure levels in `pgrb2.0p25`.
  `PWAT` is **not** on pressure levels: it exists only as the single record
  `":PWAT:entire atmosphere (considered as a single layer):anl:"`. Searching `":PWAT:850 mb:"`
  returns nothing rather than erroring.
- **Longitude is 0–360.** Convert before subsetting: `lon180 = np.where(lon > 180, lon - 360, lon)`.
  Slicing `-110 … -70` against a 0–360 axis returns an empty selection.
- **Latitude is DESCENDING** in the GRIB record order. `scipy.interpolate.RegularGridInterpolator`
  requires strictly ascending coordinates. Passing descending latitude does not raise — it returns
  values that are wrong in a way that looks plausible. Flip to ascending (`lat[::-1]`, and the data
  cube with it) before constructing any interpolator.
- **Pressure levels are stored decreasing with array index**, so `np.trapezoid` integrating over
  pressure produces a negative `dp` and the moisture-flux integral comes out with the wrong sign.
  The fix is a `* -1.0`, but a sign flip you apply from memory is a sign flip you will eventually get
  backwards. We validate it explicitly against the 850 hPa wind direction in Section 4.

In [ ]:
import os
import pickle
import warnings

import numpy as np
import pandas as pd
import requests
import io

import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.interpolate import RegularGridInterpolator
from scipy import stats

import cmweather  # noqa: F401  - registers the cmweather colormaps used across this notebook series

warnings.filterwarnings("ignore")

PC = ccrs.PlateCarree()
G0 = 9.81                      # m s-2, for the mass-weighted vapour integral
INFLOW = (41.88, -88.08)       # Glen Ellyn, IL - the first tornado of the day
T_ARRIVE = "2026-07-27 12:00:00"

### A note on colormaps

`cmweather` is imported for consistency with the radar notebooks in this series, where
`ChaseSpectral` and `HomeyerRainbow` do the work. The scalar fields here call for different
choices and both are deliberate:

- **Soil moisture uses `BrBG`** — brown-to-blue-green, diverging, and the dry/wet semantics of the
  two ends are immediately readable without consulting the colorbar.
- **IVT and evaporative fraction use `YlGnBu`** — sequential with a genuinely *pale* low end.
  This matters more than it sounds: **59% of the IVT domain falls below 150 kg m⁻¹ s⁻¹**, so a
  colormap with a dark or saturated low end fills three-fifths of the map with ink and buries the
  coastlines and state boundaries that let a reader locate the plume. We additionally leave values
  under 150 unshaded.

In [ ]:
def apply_figure_style(*, sizes=(10, 9, 8), grid=False):
    """Publication defaults: open frame, left-aligned titles, no legend box."""
    base, secondary, tick = sizes
    plt.rcParams.update({
        "font.family": "sans-serif", "font.size": base,
        "axes.labelsize": base, "axes.titlesize": base, "legend.fontsize": secondary,
        "xtick.labelsize": tick, "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "xtick.direction": "out", "ytick.direction": "out",
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": bool(grid), "legend.frameon": False,
        "figure.dpi": 200, "savefig.dpi": 300, "savefig.bbox": "tight",
        "axes.titleweight": "normal", "axes.titlelocation": "left",
        "lines.linewidth": 1.2, "pdf.fonttype": 42, "ps.fonttype": 42,
    })


META_GREY = "#5a5a5a"
TCOL = {925: "#c0201a", 850: "#1f4e8c", 700: "#7a5c00"}   # one colour per trajectory level
apply_figure_style()

## 3. Fetching the moisture fields

`fetch_moisture.py`, reproduced below, pulls the pressure-level humidity and wind plus column PWAT
for every 6-hourly analysis in the window and caches them to `gfs_moisture.pkl`. It is a download
script, not an analysis step — run it once. The seventeen levels from 1000 to 300 hPa carry
essentially all of the atmospheric water vapour, so the integral is not sensitive to extending it
higher.

Note the `sub_ll` helper: it is where the 0–360 longitude conversion happens, and it returns the
index arrays (`ji`, `ii`) so that every subsequent field is clipped to exactly the same grid without
re-deriving it.

In [ ]:
"""GFS 0.25 deg analysis fields for moisture-source diagnosis of the 27 Jul 2026 event.

Domain spans the Gulf of Mexico to the Great Lakes so integrated vapour transport
and back trajectories can attribute the moisture to a source region.
Window: 23 Jul -> 28 Jul at 6-h analysis steps.

Run once; writes gfs_moisture.pkl.
"""
from herbie import Herbie

SAVE = os.path.join(os.getcwd(), "herbie_data")

LAT0, LAT1 = 15.0, 52.0
LON0, LON1 = -110.0, -70.0
LEVS = [1000, 975, 950, 925, 900, 850, 800, 750, 700,
        650, 600, 550, 500, 450, 400, 350, 300]
TIMES = pd.date_range("2026-07-23 00:00", "2026-07-28 00:00", freq="6h")


def sub_ll(ds):
    """GFS is 0..360 lon. Return index arrays and the clipped 1-D coordinates."""
    la = ds.latitude.values
    lo = ds.longitude.values
    lo180 = np.where(lo > 180, lo - 360, lo)          # GOTCHA 1: 0-360 -> -180..180
    ji = np.where((la >= LAT0) & (la <= LAT1))[0]
    ii = np.where((lo180 >= LON0) & (lo180 <= LON1))[0]
    return ji, ii, la[ji], lo180[ii]


out = {"levels": np.array(LEVS, dtype="float32"), "times": [str(t) for t in TIMES]}
geo_done = False

for t in TIMES:
    H = Herbie(t, model="gfs", product="pgrb2.0p25", fxx=0, save_dir=SAVE, verbose=False)
    rec = {}
    lev_re = "|".join(str(L) for L in LEVS)
    for key, srch in [("q", rf":SPFH:({lev_re}) mb:"),
                      ("u", rf":UGRD:({lev_re}) mb:"),
                      ("v", rf":VGRD:({lev_re}) mb:")]:
        res = H.xarray(srch, remove_grib=False)
        ds = res[0] if isinstance(res, list) else res
        var = list(ds.data_vars)[0]
        lev = ds.isobaricInhPa.values
        order = [int(np.where(lev == L)[0][0]) for L in LEVS]   # force our level order
        if not geo_done:
            ji, ii, lat1d, lon1d = sub_ll(ds)
            out["lat"], out["lon"] = lat1d.astype("float32"), lon1d.astype("float32")
            out["_ji"], out["_ii"] = ji, ii
            geo_done = True
        arr = ds[var].values[order][:, out["_ji"], :][:, :, out["_ii"]]
        rec[key] = arr.astype("float32")

    # GOTCHA: PWAT exists ONLY as the whole-column record, not on pressure levels
    res = H.xarray(":PWAT:entire atmosphere", remove_grib=False)
    dsp = res[0] if isinstance(res, list) else res
    vp = list(dsp.data_vars)[0]
    rec["pwat"] = dsp[vp].values[out["_ji"], :][:, out["_ii"]].astype("float32")

    out[str(t)] = rec
    print(f"{t:%m-%d %HZ} ok  q{rec['q'].shape}", flush=True)

pickle.dump(out, open("gfs_moisture.pkl", "wb"))

In [ ]:
GM = pickle.load(open("gfs_moisture.pkl", "rb"))
GLEV = GM["levels"]                 # hPa, DECREASING with index
glat = GM["lat"]                    # DESCENDING
glon = GM["lon"]
GT = sorted(k for k in GM if k.startswith("2026"))

p_pa = GLEV * 100.0
LO2, LA2 = np.meshgrid(glon, glat)

print("levels (hPa):", GLEV.astype(int))
print("lat range %.2f -> %.2f  (descending: %s)" % (glat[0], glat[-1], glat[0] > glat[-1]))
print("analysis times:", len(GT), GT[0], "->", GT[-1])

## 4. Integrated vapour transport, and validating its sign

IVT is the vertically integrated horizontal moisture flux,

$$\mathrm{IVT} = \frac{1}{g}\left|\int_{p_{\mathrm{sfc}}}^{300\,\mathrm{hPa}} q\,\mathbf{V}\,dp\right|$$

with units kg m⁻¹ s⁻¹. Because the level array runs from 1000 hPa down to 300 hPa, `np.trapezoid`
sees a *decreasing* abscissa and returns a negative integral; the `* -1.0` restores the physical
sign. This is the single easiest place in the whole notebook to produce a beautiful, wrong figure —
a flipped IVT vector field points the plume in the opposite direction and still looks entirely
credible.

So we validate rather than trust. The 850 hPa wind at the inflow point and the IVT vector at the
same point must point the same way, because in this event most of the transport is in the lower
troposphere. The check below computes the angle between them.

In [ ]:
def ivt_fields(tkey):
    """Surface-300 hPa moisture flux components and magnitude at one analysis time.

    The * -1.0 corrects for GFS storing pressure levels in DECREASING order,
    which makes np.trapezoid's implicit dp negative.
    """
    d = GM[tkey]
    qu = np.trapezoid(d["q"] * d["u"], p_pa, axis=0) / G0 * -1.0
    qv = np.trapezoid(d["q"] * d["v"], p_pa, axis=0) / G0 * -1.0
    return qu, qv, np.hypot(qu, qv)


QU, QV, IVT = ivt_fields(T_ARRIVE)

In [ ]:
# --- SIGN VALIDATION: the IVT vector must be roughly parallel to the 850 hPa wind ---
j = int(np.argmin(np.abs(glat - INFLOW[0])))
i = int(np.argmin(np.abs(glon - INFLOW[1])))
k850 = int(np.argmin(np.abs(GLEV - 850)))

u850 = float(GM[T_ARRIVE]["u"][k850, j, i])
v850 = float(GM[T_ARRIVE]["v"][k850, j, i])

bearing = lambda u, v: (np.degrees(np.arctan2(u, v)) % 360.0)
b_wind, b_ivt = bearing(u850, v850), bearing(float(QU[j, i]), float(QV[j, i]))
misfit = abs((b_ivt - b_wind + 180.0) % 360.0 - 180.0)

print(f"850 hPa wind toward {b_wind:6.1f} deg   IVT vector toward {b_ivt:6.1f} deg"
      f"   misfit {misfit:5.1f} deg")
assert misfit < 60.0, "IVT sign is inverted - check the trapezoid dp direction"

# fraction of the domain below the atmospheric-river-adjacent 150 kg/m/s threshold,
# which is why we mask it out and use a pale-low-end colormap
print("domain fraction below 150 kg/m/s: %.0f%%" % (100 * np.mean(IVT < 150)))

**Result.** IVT at the Glen Ellyn inflow point, taken from the higher-resolution HRRR analysis at
17 UTC (the hour of the tornadoes), was **822 kg m⁻¹ s⁻¹**. For scale: atmospheric-river
classification schemes typically begin around **250 kg m⁻¹ s⁻¹**, and a typical summer Midwest value
is **200–400**. The transport feeding these storms was two to four times a normal July value and
comfortably in atmospheric-river territory — a framing usually reserved for West Coast winter
events, but the metric does not care about the season.

## 5. Which part of the column matters?

Before running trajectories we need to know *which level to believe*. Trajectories at different
levels will arrive from different places, and the honest way to choose between them is to ask where
the water actually is. We partition the column precipitable water into layers:

$$\mathrm{PW}_{\mathrm{layer}} = \frac{1}{g\rho_w}\int q\,dp$$

In [ ]:
def layer_pw(tkey, jj, ii, p_top, p_bot):
    """Precipitable water in mm between two pressure levels at one grid point."""
    q = GM[tkey]["q"][:, jj, ii]
    m = (GLEV <= p_bot) & (GLEV >= p_top)
    return float(np.trapezoid(q[m], p_pa[m]) / G0 * -1.0)   # kg m-2 == mm


bands = [(850, 1000), (700, 850), (500, 700), (300, 500)]
pw = {b: layer_pw(T_ARRIVE, j, i, *b) for b in bands}
total = sum(pw.values())
for (pt, pb), v in pw.items():
    print(f"{pb:4.0f}-{pt:4.0f} hPa : {v:5.1f} mm  ({100*v/total:4.1f}% of column)")
print(f"{'total':13s}: {total:5.1f} mm")

**Result.** Of a **66.1 mm** column, **28.3 mm — 43% — sat below 850 hPa**; 20.9 mm in 850–700,
14.0 mm in 700–500, and only 2.9 mm above 500 hPa. Two consequences follow.

First, **the 925 hPa trajectory is the storm-relevant one.** That is the layer the updraught was
ingesting and it is where nearly half the water was. A 700 hPa back trajectory is a real answer to a
different question.

Second, the dryness aloft is not a defect of the moisture supply — **it is part of the severe-weather
recipe.** A very moist boundary layer under a comparatively dry mid-troposphere steepens the lapse
rate and increases CAPE relative to a uniformly moist column. The vertical *structure* of the
moisture mattered as much as its amount.

## 6. Isobaric back trajectories

We integrate parcels backward from the Glen Ellyn inflow point (41.88 °N, 88.08 °W) at 12 UTC on
27 July, for 96 hours, at 1-hour steps, using second-order Runge–Kutta (midpoint) on the GFS wind
field interpolated in time, latitude and longitude. Specific humidity is sampled along the way from
the same interpolated field.

The trajectories are **isobaric** — parcels are advected along a constant pressure surface rather
than following true three-dimensional motion. Section 10 discusses what that costs us.

### Warning: flip latitude before building the interpolator

`RegularGridInterpolator` demands strictly ascending coordinates. GFS latitude is descending. If you
hand it the raw arrays it does not complain; it returns numbers, and those numbers are wrong. Both
the coordinate array *and* the data cube must be reversed together — note the `[::-1, :]` inside
`make_interp` matching `lat_a = glat[::-1]` outside it.

In [ ]:
lat_a = glat[::-1]                                    # GOTCHA: ascending for the interpolator
tnum = np.array([pd.Timestamp(t).value / 1e9 for t in GT])   # seconds since epoch


def make_interp(field_key, plev):
    """Trilinear (t, lat, lon) interpolator for one field at the nearest pressure level."""
    k = int(np.argmin(np.abs(GLEV - plev)))
    cube = np.stack([GM[t][field_key][k][::-1, :] for t in GT])   # flip lat WITH the data
    return RegularGridInterpolator((tnum, lat_a, glon), cube,
                                   bounds_error=False, fill_value=np.nan)


def back_traj(lat0, lon0, plev, t_end, hours=96, dt_h=1.0):
    """RK2 (midpoint) backward isobaric trajectory. Returns (time, lat, lon, q in g/kg)."""
    fu, fv, fq = (make_interp(k, plev) for k in ("u", "v", "q"))
    t = pd.Timestamp(t_end).value / 1e9
    la, lo = lat0, lon0
    T, LA, LO = [t], [la], [lo]
    Q = [float(fq([[t, la, lo]])[0])]

    for _ in range(int(hours / dt_h)):
        u1 = float(fu([[t, la, lo]])[0])
        v1 = float(fv([[t, la, lo]])[0])
        if not np.isfinite(u1):
            break
        dt = dt_h * 3600.0
        # half step backward to the midpoint
        la_m = la - v1 * dt / 2 / 111000.0
        lo_m = lo - u1 * dt / 2 / (111000.0 * np.cos(np.radians(la)))
        u2 = float(fu([[t - dt / 2, la_m, lo_m]])[0])
        v2 = float(fv([[t - dt / 2, la_m, lo_m]])[0])
        if not np.isfinite(u2):
            u2, v2 = u1, v1
        # full step backward using the midpoint wind
        la = la - v2 * dt / 111000.0
        lo = lo - u2 * dt / (111000.0 * np.cos(np.radians(la)))
        t -= dt
        if not (glat.min() < la < glat.max() and glon.min() < lo < glon.max()):
            break
        T.append(t); LA.append(la); LO.append(lo)
        Q.append(float(fq([[t, la, lo]])[0]))

    times = np.array([pd.Timestamp(int(x * 1e9)) for x in T])
    return times, np.array(LA), np.array(LO), np.array(Q) * 1000.0   # kg/kg -> g/kg

In [ ]:
TRAJ = {}
for plev in (925, 850, 700):
    TRAJ[plev] = back_traj(*INFLOW, plev, T_ARRIVE, hours=96)

for plev, (tt, la, lo, q) in TRAJ.items():
    print(f"{plev:4d} hPa  origin {la[-1]:6.2f} N {abs(lo[-1]):6.2f} W  "
          f"{len(la)-1:3d} h back   q {q[-1]:5.1f} -> {q[0]:5.1f} g/kg  "
          f"({100*(q[0]/q[-1]-1):+.0f}%)")

**Result — and it is level-dependent, which is the point.**

| Level | Position 96 h earlier (23 Jul) | Source region |
|---|---|---|
| **925 hPa** | **29.43 °N, 97.20 °W** | **Texas coast / NW Gulf of Mexico** |
| 850 hPa | 36.81 °N, 101.74 °W | southern High Plains |
| 700 hPa | 46.45 °N, 109.02 °W | Montana |

Only the **low levels** were Gulf-sourced. The 925 hPa parcel started over the northwest Gulf coast
and **gained 44% more vapour en route, from 12.5 to 18.1 g kg⁻¹**. Meanwhile the air at 700 hPa had
come across the northern Rockies — continental, and comparatively dry.

That vertical contrast is exactly the structure Section 5 inferred from the PW partition, arrived at
independently. A single-level trajectory would have supported whichever story you happened to plot.
Running three and reporting all three is what makes the claim about the *low-level* Gulf source
defensible.

## 7. An independent observational check: the O'Hare dewpoint surge

Everything so far comes from the GFS humidity field. If the model's moisture were wrong, the whole
argument would be wrong in the same direction and we would not know. So we check the arrival against
a gauge: the ASOS dewpoint record at Chicago O'Hare, from the Iowa Environmental Mesonet.

In [ ]:
IEM = "https://mesonet.agron.iastate.edu/cgi-bin/request"

# hourly temperature / dewpoint through the event week
u_hourly = (f"{IEM}/asos.py?station=ORD&data=tmpf&data=dwpf&tz=UTC"
            "&sts=2026-07-20T00:00Z&ets=2026-07-28T00:00Z&format=onlycomma&missing=M")
hh = pd.read_csv(io.StringIO(requests.get(u_hourly, timeout=180).text), na_values=["M", ""])
hh["valid"] = pd.to_datetime(hh.valid)
daily = hh.dropna(subset=["dwpf"]).set_index("valid").dwpf.resample("1D").agg(["max", "mean"])

# full daily climate record for the percentile and the July normal
u_daily = (f"{IEM}/daily.py?network=IL_ASOS&stations=ORD"
           "&year1=1990&month1=1&day1=1&year2=2026&month2=8&day2=1&format=csv")
dl = pd.read_csv(io.StringIO(requests.get(u_daily, timeout=240).text), na_values=["M", "None"])
dl["day"] = pd.to_datetime(dl.day)

july = dl[dl.day.dt.month == 7].dropna(subset=["max_dewpoint_f"])
jd_ser = july.groupby(july.day.dt.year)["max_dewpoint_f"].mean()   # per-year July mean
normal = float(jd_ser.loc[1991:2020].mean())                       # 1991-2020 July normal

hist = july[july.day.dt.year < 2026]["max_dewpoint_f"]
obs = float(daily.loc["2026-07-27", "max"])
n_at_or_above = int((hist >= obs).sum())

print(f"23 Jul max dewpoint {daily.loc['2026-07-23','max']:.0f} F"
      f" -> 27 Jul {obs:.0f} F   surge {obs - daily.loc['2026-07-23','max']:+.0f} F")
print(f"1991-2020 July normal daily-max dewpoint: {normal:.1f} F")
print(f"percentile of {obs:.0f} F among {len(hist)} July days since 1990: "
      f"{100*(hist < obs).mean():.1f}th  ({n_at_or_above} days at or above)")

**Result.** The O'Hare daily maximum dewpoint rose from **48 °F on 23 July to 79 °F on 27 July —
a 31 °F surge in four days**, tracking the trajectory's arrival. The 1991–2020 July normal is
66.6 °F. A 79 °F daily-max dewpoint sits at the **99.7th percentile of every July day since 1990**:
only **14 of 1147 days** reached it, about **0.4 days in a typical July**.

The GFS moistening along the 925 hPa path and the observed dewpoint surge at the destination are
independent measurements of the same event, and they agree. That is what licenses the trajectory
interpretation.

## 8. Figure 1 — moisture transport and back trajectories

Three panels: the trajectory bundle over the IVT field at arrival time, the 925 hPa parcel's
moisture history, and the observed dewpoint surge at O'Hare with the July normal for reference.

In [ ]:
def build_moisture_figure():
    apply_figure_style()
    F = plt.figure(figsize=(15.0, 10.4))

    # ---------------- (a) IVT map with the trajectory bundle ----------------
    A = F.add_axes([0.075, 0.400, 0.615, 0.470], projection=PC)
    A.set_extent([-104, -74, 20, 49], crs=PC)
    A.add_feature(cfeature.LAND.with_scale("50m"), fc="#f7f5f0", zorder=0)
    A.add_feature(cfeature.OCEAN.with_scale("50m"), fc="#e4eff5", zorder=0)
    A.add_feature(cfeature.STATES.with_scale("50m"), ec="0.72", lw=0.35, zorder=3)
    A.add_feature(cfeature.COASTLINE.with_scale("50m"), ec="0.2", lw=0.7, zorder=3)
    A.add_feature(cfeature.BORDERS.with_scale("50m"), ec="0.3", lw=0.6, zorder=3)

    ivt_m = np.where(IVT < 150, np.nan, IVT)      # leave the quiet 59% of the domain unshaded
    cf = A.contourf(LO2, LA2, ivt_m, levels=[150, 200, 250, 300, 400, 500, 650, 800, 1000, 1400],
                    cmap="YlGnBu", extend="max", transform=PC, zorder=1, alpha=0.85)
    cax = F.add_axes([0.700, 0.400, 0.014, 0.470])
    cb = F.colorbar(cf, cax=cax)
    cb.set_label("Integrated vapour transport (kg m$^{-1}$ s$^{-1}$)", fontsize=8.5)
    cb.ax.tick_params(labelsize=7.5)

    sk = 5
    A.quiver(LO2[::sk, ::sk], LA2[::sk, ::sk], QU[::sk, ::sk], QV[::sk, ::sk], transform=PC,
             scale=1.3e4, width=0.0013, color="0.35", alpha=0.65, zorder=4)

    for plev in (700, 850, 925):        # draw 925 last so it sits on top
        _, la_, lo_, _ = TRAJ[plev]
        A.plot(lo_, la_, color=TCOL[plev], lw=2.6, transform=PC, zorder=6,
               path_effects=[pe.withStroke(linewidth=4.4, foreground="white")])
        A.plot(lo_[-1], la_[-1], marker="o", ms=9, mfc=TCOL[plev], mec="white", mew=1.5,
               transform=PC, zorder=7)
        for h in range(24, 97, 24):     # 24-hour tick marks along each path
            if h < len(la_):
                A.plot(lo_[h], la_[h], marker="o", ms=4.2, mfc="white", mec=TCOL[plev], mew=1.3,
                       transform=PC, zorder=7)

    A.plot(INFLOW[1], INFLOW[0], marker="*", ms=19, mfc="#111111", mec="white", mew=1.4,
           transform=PC, zorder=8)
    A.text(-87.2, 42.7, "Chicago", fontsize=9, fontweight="bold", transform=PC, zorder=8,
           path_effects=[pe.withStroke(linewidth=2.4, foreground="white")])
    A.text(-97.4, 27.6, "Texas / NW Gulf\n4 days earlier", fontsize=8, color=TCOL[925],
           fontweight="bold", ha="center", va="top", transform=PC, zorder=8,
           path_effects=[pe.withStroke(linewidth=2.4, foreground="white")])

    handles = [Line2D([], [], color=TCOL[p], lw=2.6, label=f"{p} hPa parcel path")
               for p in (925, 850, 700)]
    handles.append(Line2D([], [], marker="o", ms=6, mfc="white", mec="0.3", ls="",
                          label="24-hour marks"))
    A.legend(handles=handles, loc="lower left", fontsize=7.8, framealpha=0.94)
    A.set_title("(a)  96-hour back trajectories into the storm inflow, over vapour transport "
                "at 27 Jul 12 UTC", loc="left", fontsize=10.2, pad=7)
    gl = A.gridlines(draw_labels=True, lw=0.3, color="0.75", alpha=0.5, zorder=2)
    gl.top_labels = gl.right_labels = False
    gl.xlabel_style = gl.ylabel_style = {"size": 7}

    # ---------------- (b) moisture history of the 925 hPa parcel ----------------
    B = F.add_axes([0.075, 0.062, 0.365, 0.250])
    _, _, _, q_ = TRAJ[925]
    ageh = -np.arange(len(q_))
    B.plot(ageh, q_, color=TCOL[925], lw=2.2, zorder=3)
    B.fill_between(ageh, 5, q_, color=TCOL[925], alpha=0.13, zorder=2)
    B.axhline(q_[-1], color="0.45", ls=":", lw=1.0, zorder=1)
    B.annotate(f"leaving the Gulf coast\n{q_[-1]:.1f} g kg$^{{-1}}$", xy=(-94, q_[-1]),
               xytext=(-84, 6.6), fontsize=7.6, color="#8a1a16",
               arrowprops=dict(arrowstyle="->", lw=0.9, color="#8a1a16"))
    B.annotate(f"arriving over Chicago\n{q_[0]:.1f} g kg$^{{-1}}$", xy=(-1, q_[0]),
               xytext=(-44, 21.2), fontsize=7.6, color="#8a1a16", fontweight="bold",
               arrowprops=dict(arrowstyle="->", lw=0.9, color="#8a1a16"))
    B.set_xlabel("Hours before arrival over Chicago")
    B.set_ylabel("Specific humidity (g kg$^{-1}$)")
    B.set_xlim(-96, 4); B.set_ylim(5, 23.5); B.grid(alpha=0.3, zorder=0)
    B.set_title(f"(b)  The 925 hPa parcel gained {(q_[0]/q_[-1]-1)*100:.0f}% more vapour en route",
                loc="left", fontsize=10.2, pad=7)

    # ---------------- (c) observed dewpoint surge at O'Hare ----------------
    D = F.add_axes([0.520, 0.062, 0.365, 0.250])
    dd = daily.loc["2026-07-20":"2026-07-27"]
    x = np.arange(len(dd))
    D.bar(x, dd["max"], color=["#b8c4d0"] * (len(dd) - 1) + [TCOL[925]], zorder=3, width=0.68)
    D.axhline(normal, color="#1a7a3d", ls="--", lw=1.4, zorder=4)
    D.text(0.03, 0.62, f"dashed line = 1991\u20132020\nJuly normal, {normal:.1f} \u00b0F",
           transform=D.transAxes, fontsize=7.2, color="#1a7a3d", fontweight="bold",
           ha="left", va="bottom")
    for xi, v in zip(x, dd["max"]):
        D.text(xi, v + 1.0, f"{v:.0f}", ha="center", fontsize=7.6,
               fontweight="bold" if xi == x[-1] else "normal",
               color=TCOL[925] if xi == x[-1] else "0.3")
    D.set_xticks(x)
    D.set_xticklabels([f"{d_:%-d}" for d_ in dd.index], fontsize=8)
    D.set_xlabel("Day of July 2026")
    D.set_ylabel("Daily max dewpoint, O'Hare (\u00b0F)")
    D.set_ylim(40, 96); D.grid(axis="y", alpha=0.3, zorder=0)
    D.annotate("+31 \u00b0F in four days", xy=(6.55, 80.5), xytext=(1.9, 90.0),
               fontsize=8.4, color=TCOL[925], fontweight="bold",
               arrowprops=dict(arrowstyle="->", lw=1.0, color=TCOL[925]))
    D.set_title("(c)  79 \u00b0F \u2014 99.7th percentile of every July day since 1990",
                loc="left", fontsize=10.2, pad=7)

    F.suptitle("Where the moisture came from \u2014 27 July 2026 Chicagoland tornado outbreak",
               x=0.075, ha="left", fontsize=13.5, y=0.982)
    F.text(0.075, 0.944,
           "GFS 0.25\u00b0 analyses. Vapour transport is the surface\u2013300 hPa moisture flux; "
           "above ~250 kg m$^{-1}$ s$^{-1}$ is atmospheric-river magnitude (values under 150 left "
           "unshaded).\nTrajectories are isobaric, integrated backward from the Glen Ellyn storm "
           "inflow. 43% of the column moisture sat below 850 hPa, so the 925 hPa path is the "
           "storm-relevant one.",
           fontsize=8.4, color=META_GREY, ha="left", va="top")
    return F


figM = build_moisture_figure()
figM.savefig("moisture_source.png", dpi=200, bbox_inches="tight")

![Moisture source: 96-hour isobaric back trajectories from the Glen Ellyn inflow over GFS integrated vapour transport at 27 Jul 12 UTC; 925 hPa parcel moisture history; observed O'Hare daily maximum dewpoint 20-27 July 2026](02_moisture_source_and_transport_outputs/moisture_source.png)

Read panel (a) from the bottom up. The Gulf plume is continuous from the Texas coast to the western
Great Lakes, and the shaded IVT maximum sits directly over the tornado corridor. The three
trajectories diverge dramatically going backward in time: only the red 925 hPa path reaches the
Gulf. Panel (b) shows the moistening is not a single jump at the coast but a gradual accumulation
with a mid-path setback — which is what Section 9 is about. Panel (c) is the observation that keeps
the whole thing honest.

## 9. The land surface along the path

Panel (b) of Figure 1 shows the parcel did not moisten monotonically. That prompts the question that
drives the rest of this notebook, and it is a question with an intuitive answer that turns out to be
wrong:

> *Was the path nice and wet, so the parcel held onto its Gulf moisture the whole way?*

The intuition is that a saturated surface protects a moist plume — no dry-soil sink, and evaporation
adds more. The way to test it is the surface energy budget. Latent heat flux `LHTFL` is energy going
into evaporation; sensible heat flux `SHTFL` is energy going into warming the air. Their ratio,

$$\mathrm{EF} = \frac{LH}{LH + SH}, \qquad \mathrm{Bowen} = \frac{SH}{LH}$$

tells you which the land was doing. High EF (low Bowen) means the surface was pumping vapour into the
boundary layer; low EF (high Bowen) means it was heating and drying it.

### Warning: surface flux timing will bite you

`LHTFL` and `SHTFL` in GFS GRIB are **time-averaged over the forecast interval**, not instantaneous.
The averaging window is therefore a choice you are making whether you realise it or not, and getting
it wrong produces a physically coherent, completely useless answer.

The first version of this analysis used the **06 UTC cycle at `fxx=6`**, an obvious-looking choice.
That window is 06–12 UTC = **1–7 am CDT**: nocturnal. Sensible heat came back negative (the ground
cooler than the air, heat flowing downward) and latent heat came back near zero. Evaporative fraction
computed from those numbers is meaningless.

The correct choice for daytime land–atmosphere coupling is the **12 UTC cycle at `fxx=9`**, whose
6–9 hour average covers **18–21 UTC = local afternoon over the Plains** — peak insolation, peak
evapotranspiration. That is the window used below.

### Warning: two more, on SOILW and on EF

- **`SOILW` returns as ONE dataset with all four soil layers stacked** on a `depthBelowLandLayer`
  dimension, shape `(4, lat, lon)` — not as four separate datasets, which is what a `:SOILW:` search
  string leads you to expect. Any clip/subset helper must handle `ndim == 3`. The `clip()` function
  below branches on exactly that.
- **Evaporative fraction can exceed 1** wherever `SH` is slightly **negative** — downward sensible
  heat over a wet surface cooler than the air overhead. That is physically real, and it is common
  over saturated ground on a humid afternoon, but it makes the ratio ill-behaved near the singularity
  at `LH + SH = 0`. We therefore (i) require `LH + SH` above a small threshold before computing EF at
  all, (ii) report the **Bowen ratio alongside** because it degrades gracefully, and (iii) clip EF to
  [0, 1] **for display only**, never before averaging.

In [ ]:
"""GFS land-surface fields along the 27 Jul 2026 moisture advection path.

Soil moisture (SOILW) and the land mask come from the fxx=0 analysis at 12Z.
Turbulent fluxes are TIME-AVERAGED, so they come from the 12Z cycle at fxx=9,
whose 6-9 h average window is 18-21 UTC = local afternoon over the Plains.

Run once; writes gfs_land.pkl and gfs_flx.pkl.
"""
DAYS = pd.date_range("2026-07-23", "2026-07-27", freq="1D")

land_out = {"days": [str(d.date()) for d in DAYS]}
geo = False


def clip(ds):
    """Subset one Herbie dataset to the analysis domain, converting lon to -180..180.

    Handles ndim == 3 because SOILW arrives with all four soil layers stacked
    on depthBelowLandLayer.
    """
    global geo
    la = ds.latitude.values
    lo180 = np.where(ds.longitude.values > 180, ds.longitude.values - 360, ds.longitude.values)
    ji = np.where((la >= LAT0) & (la <= LAT1))[0]
    ii = np.where((lo180 >= LON0) & (lo180 <= LON1))[0]
    if not geo:
        land_out["lat"] = la[ji].astype("float32")
        land_out["lon"] = lo180[ii].astype("float32")
        land_out["_ji"], land_out["_ii"] = ji, ii
        geo = True
    a = ds[list(ds.data_vars)[0]].values
    if a.ndim == 3:                                        # (level, lat, lon) - stacked soil layers
        return a[:, land_out["_ji"], :][:, :, land_out["_ii"]].astype("float32")
    return a[land_out["_ji"], :][:, land_out["_ii"]].astype("float32")


for d in DAYS:
    rec = {}
    Ha = Herbie(d + pd.Timedelta(hours=12), model="gfs", product="pgrb2.0p25",
                fxx=0, save_dir=SAVE, verbose=False)

    res = Ha.xarray(":SOILW:", remove_grib=False)
    sm = {}
    for ds in (res if isinstance(res, list) else [res]):
        arr = clip(ds)
        tops = np.atleast_1d(ds.depthBelowLandLayer.values).astype(float)
        if arr.ndim == 3:
            for k, top in enumerate(tops):
                sm[round(float(top), 2)] = arr[k]
        else:
            sm[round(float(tops[0]), 2)] = arr
    rec["soilw"] = sm

    for key, srch in [("land", ":LAND:surface:"), ("veg", ":VEG:surface:")]:
        r = Ha.xarray(srch, remove_grib=False)
        rec[key] = clip(r[0] if isinstance(r, list) else r)

    land_out[str(d.date())] = rec
    print(f"{d.date()} soil layers {sorted(sm)}", flush=True)

pickle.dump(land_out, open("gfs_land.pkl", "wb"))

# --- daytime-averaged turbulent fluxes: 12Z cycle, fxx=9 -> 18-21 UTC average ---
flx_out = {}
for d in DAYS:
    Hf = Herbie(d + pd.Timedelta(hours=12), model="gfs", product="pgrb2.0p25",
                fxx=9, save_dir=SAVE, verbose=False)
    rec = {}
    for key, srch in [("lhtfl", ":LHTFL:surface:"), ("shtfl", ":SHTFL:surface:")]:
        r = Hf.xarray(srch, remove_grib=False)
        rec[key] = clip(r[0] if isinstance(r, list) else r)
    flx_out[str(d.date())] = rec
    print(f"{d.date()} fluxes ok", flush=True)

pickle.dump(flx_out, open("gfs_flx.pkl", "wb"))

In [ ]:
LD = pickle.load(open("gfs_land.pkl", "rb"))
FLX = pickle.load(open("gfs_flx.pkl", "rb"))

llat, llon = LD["lat"], LD["lon"]
LO3, LA3 = np.meshgrid(llon, llat)
onland = LD[LD["days"][0]]["land"] > 0.5

print("soil layer tops (m):", sorted(LD["2026-07-27"]["soilw"]))
print("land fraction of domain: %.0f%%" % (100 * onland.mean()))

### Sampling the surface along the 925 hPa path

For each hourly trajectory point we take the nearest land grid cell and record 0–10 cm and 10–40 cm
soil moisture, the two turbulent fluxes, the derived EF and Bowen ratio, and the parcel's specific
humidity. Nearest-neighbour rather than interpolation is deliberate here: soil moisture and surface
flux fields are genuinely discontinuous at coastlines and land-use boundaries, and interpolating
across them invents values.

In [ ]:
_, la_, lo_, q_ = TRAJ[925]
tt_ = TRAJ[925][0]

rows = []
for k in range(len(la_) - 1, -1, -1):          # walk forward in time (oldest point first)
    t = pd.Timestamp(tt_[k])
    dkey = str(t.date())
    if dkey not in LD or dkey not in FLX:
        continue
    jj = int(np.argmin(np.abs(llat - la_[k])))
    ii = int(np.argmin(np.abs(llon - lo_[k])))
    lh = float(FLX[dkey]["lhtfl"][jj, ii])
    sh = float(FLX[dkey]["shtfl"][jj, ii])
    ef = lh / (lh + sh) if (lh + sh) > 20 else np.nan     # guard the singularity
    rows.append(dict(hours_before=-k, valid=str(t),
                     lat=round(float(la_[k]), 2), lon=round(float(lo_[k]), 2),
                     soilw_0_10cm=round(float(LD[dkey]["soilw"][0.0][jj, ii]), 3),
                     soilw_10_40cm=round(float(LD[dkey]["soilw"][0.1][jj, ii]), 3),
                     LH_Wm2=round(lh, 1), SH_Wm2=round(sh, 1),
                     evap_fraction=round(ef, 3) if np.isfinite(ef) else np.nan,
                     q_g_kg=round(float(q_[k]), 2)))

LP = pd.DataFrame(rows)
LP["bowen"] = (LP.SH_Wm2 / LP.LH_Wm2).round(2)
LP["ef_clip"] = LP.evap_fraction.clip(0, 1)      # clipped for DISPLAY, after the raw value is kept

print(LP[["hours_before", "lat", "lon", "soilw_0_10cm", "evap_fraction", "bowen", "q_g_kg"]]
      .iloc[::12].to_string(index=False))

## 10. An appealing relationship that did not survive testing

The obvious next move is a scatter plot: hourly parcel moisture change `dq` against hourly
evaporative fraction, with a fitted line and an r value. If wet, evaporating ground moistens the
parcel, that scatter should slope upward, and it would be a satisfying single-panel result.

We ran it. **It does not hold up**, and the reason is instructive rather than disappointing.

In [ ]:
# The version of this panel that was CUT. Kept in the notebook deliberately.
dq_hourly = LP.q_g_kg.diff()
mask = np.isfinite(dq_hourly) & np.isfinite(LP.ef_clip)
r_h, p_h = stats.pearsonr(LP.ef_clip[mask], dq_hourly[mask])
print(f"hourly dq vs EF:  r = {r_h:+.2f}   p = {p_h:.2f}   n = {int(mask.sum())}")

# ... and the same test on the 4-leg aggregate (computed in the next cell)

**Result: r = +0.10, p = 0.35, n = 97.** Not significant, and not publishable.

The reason is not that the physics is absent — it is that **hourly `dq` along an interpolated
trajectory is dominated by interpolation noise.** The trajectory position at any hour carries error;
the humidity field is sampled trilinearly from 6-hourly analyses; differencing that sampled series
hour-to-hour amplifies the noise relative to the ~0.1 g kg⁻¹ h⁻¹ physical signal we are trying to
detect. The scatter was measuring the interpolator, not the land surface.

So the panel was replaced with a **four-leg aggregated moisture budget**, which averages the noise
down by roughly √24 per leg and shows the real structure. For completeness: the same correlation
computed on the four aggregated legs gives **r = +0.39, p = 0.61** — a larger coefficient on four
points, and even less significant. Four points cannot support a regression.

**The leg values are therefore presented as values, not as a fitted relationship.** This is the
honest form of the result: the numbers show a clear sign change along the path, and the sign change
is the finding. Reporting "r = +0.39" with a trend line would dress up four data points as a
correlation, and the first person to check would find p = 0.61. Test the appealing relationship
before you publish it — and when it fails, report the aggregate rather than the fit.

In [ ]:
legs = [(-96, -73, "Texas"),
        (-72, -49, "Oklahoma / S. Plains"),
        (-48, -25, "Kansas / Missouri"),
        (-24,   0, "Iowa / Illinois")]

agg = []
for a, b, lab in legs:
    s = LP[(LP.hours_before >= a) & (LP.hours_before <= b)].sort_values("hours_before")
    dq_leg = float(s.q_g_kg.iloc[-1] - s.q_g_kg.iloc[0])
    hours = len(s) - 1
    agg.append(dict(leg=lab, hours=hours,
                    soilw=round(s.soilw_0_10cm.mean(), 3),
                    EF=round(s.ef_clip.mean(), 2),
                    bowen=round(s.bowen.median(), 2),
                    dq_total=round(dq_leg, 2),
                    dq_per_day=round(dq_leg / (hours / 24), 2)))
AG = pd.DataFrame(agg)
print(AG.to_string(index=False))

r_a, p_a = stats.pearsonr(AG.EF, AG.dq_per_day)
print(f"\naggregated 4-leg dq vs EF:  r = {r_a:+.2f}   p = {p_a:.2f}   n = {len(AG)}"
      "   <- reported as leg values, NOT as a regression")

**Result — and the intuitive answer was wrong.** Was the path nice and wet? **No, not uniformly.
There was a dry gap in the middle.**

| Leg | 0–10 cm soil moisture (m³ m⁻³) | Evaporative fraction | Bowen ratio | Parcel dq (g kg⁻¹ day⁻¹) |
|---|---|---|---|---|
| Texas coast → central TX | 0.165 | 0.43 | 1.26 | **+4.4** |
| Oklahoma / southern Plains | **0.106** | 0.34 | 1.89 | **−2.0** |
| Kansas / Missouri | 0.184 | 0.63 | 0.54 | −0.4 |
| Iowa / Illinois | **0.195** | **0.95** | **0.04** | **+3.2** |

The parcel **lost vapour crossing the hot dry southern Plains** — soil moisture 0.106 m³ m⁻³, Bowen
ratio near 2, the surface heating the boundary layer and deepening it so that dry air entrained from
above diluted the plume. It then **regained that vapour and more over saturated Iowa and Illinois**,
where the land was putting **about 95% of its available surface energy into evaporation rather than
into heating the air** (Bowen 0.04).

The conclusion is not the one the framing anticipated: **the wet soil at the receiving end mattered
more than wet soil at the source.** The Gulf supplied the water, but the final 24 hours over
saturated Corn Belt soil is where the inflow got its last +3.2 g kg⁻¹ per day — the increment that
took a very unstable airmass to an extremely unstable one.

### Warning: GFS soil moisture is a model state, not a measurement

`SOILW` is the **Noah land-surface model state running inside the GFS data assimilation**. It is spun
up by GFS's own precipitation history, which means that wherever GFS mis-forecast rain over the
preceding weeks, its soil moisture is wrong — and there is no observation in the loop to correct it.
Treating it as a soil-moisture observation is a category error.

This is precisely why the next section exists. An independent, gauge-based check on the same spatial
pattern is not optional garnish; it is what makes the soil-moisture argument usable.

In [ ]:
# Independent observational check: 30-day antecedent rainfall at ASOS stations near the path,
# normalised to the same calendar window (day-of-year 179-208) across 1995-2025.
PATH_STNS = [("VCT", "TX_ASOS", "Victoria TX (near parcel origin)", 28.86, -96.93),
             ("ACT", "TX_ASOS", "Waco TX",                         31.62, -97.23),
             ("OKC", "OK_ASOS", "Oklahoma City OK",                35.39, -97.60),
             ("ICT", "KS_ASOS", "Wichita KS",                      37.65, -97.43),
             ("MCI", "MO_ASOS", "Kansas City MO",                  39.30, -94.71),
             ("DSM", "IA_ASOS", "Des Moines IA",                   41.53, -93.66),
             ("ORD", "IL_ASOS", "Chicago O'Hare IL",               41.98, -87.90)]

rows = []
for sid, net, lab, la0, lo0 in PATH_STNS:
    u = (f"{IEM}/daily.py?network={net}&stations={sid}"
         "&year1=1995&month1=1&day1=1&year2=2026&month2=7&day2=28&format=csv")
    d = pd.read_csv(io.StringIO(requests.get(u, timeout=240).text), na_values=["M", "None"])
    d["day"] = pd.to_datetime(d.day)

    obs30 = d[(d.day >= "2026-06-28") & (d.day <= "2026-07-27")].precip_in.sum()

    w = d[(d.day.dt.dayofyear >= 179) & (d.day.dt.dayofyear <= 208)].dropna(subset=["precip_in"])
    per_yr = w.groupby(w.day.dt.year).precip_in.sum()
    per_yr = per_yr[per_yr.index < 2026]

    rows.append(dict(station=sid, label=lab, lat=la0, lon=lo0,
                     obs_30d=round(float(obs30), 2),
                     normal_30d=round(float(per_yr.mean()), 2),
                     pct_of_normal=round(100 * obs30 / per_yr.mean()),
                     percentile=round(100 * float((per_yr <= obs30).mean()))))

PATHN = pd.DataFrame(rows)
print(PATHN[["station", "obs_30d", "normal_30d", "pct_of_normal", "percentile"]].to_string(index=False))

**Result — the gauges corroborate the model soil pattern, including the dry gap.**
30-day rainfall as a percentage of the 1995–2025 normal for the same calendar window:

| Station | % of normal | Percentile |
|---|---|---|
| Victoria TX (near parcel origin) | 73% | 58th |
| Waco TX | 113% | 61st |
| Oklahoma City OK | 93% | 52nd |
| **Wichita KS** | **9%** | **3rd — the dry gap** |
| Kansas City MO | 47% | 16th |
| **Des Moines IA** | **220%** | **97th** |
| Chicago O'Hare IL | 138% | 84th |

Wichita received **9% of its normal** antecedent rainfall, a 3rd-percentile value, and Kansas City
47%. Des Moines received **220% of normal**, a 97th-percentile value, and O'Hare 138%. The
independent gauge record reproduces the same along-path structure as the Noah soil-moisture state:
dry in the middle, saturated at the receiving end. Two data sources with entirely different error
characteristics agreeing on the same spatial pattern is much stronger evidence than either alone.

## 11. Figure 2 — soil conditions along the moisture path

Five panels: the soil-moisture field with the 925 hPa path overlaid, the afternoon evaporative
fraction, the along-path profile of both, the observed antecedent-rainfall check, and the
four-leg moisture budget.

In [ ]:
def build_soil_figure():
    apply_figure_style()
    F = plt.figure(figsize=(15.0, 10.2))

    # ---------------- (a) soil moisture, 27 Jul, with the trajectory ----------------
    A = F.add_axes([0.055, 0.545, 0.40, 0.375], projection=PC)
    A.set_extent([-104, -82, 25, 47], crs=PC)
    A.add_feature(cfeature.OCEAN.with_scale("50m"), fc="#dceaf2", zorder=2)
    A.add_feature(cfeature.STATES.with_scale("50m"), ec="0.55", lw=0.4, zorder=3)
    A.add_feature(cfeature.COASTLINE.with_scale("50m"), ec="0.2", lw=0.7, zorder=3)

    sm = np.where(onland, LD["2026-07-27"]["soilw"][0.0], np.nan)
    cf1 = A.contourf(LO3, LA3, sm, levels=np.arange(0.02, 0.42, 0.02),
                     cmap="BrBG", extend="both", transform=PC, zorder=1)
    cax1 = F.add_axes([0.462, 0.545, 0.011, 0.375])
    cb1 = F.colorbar(cf1, cax=cax1)
    cb1.set_label("0\u201310 cm volumetric soil moisture (m\u00b3 m$^{-3}$)", fontsize=8)
    cb1.ax.tick_params(labelsize=7)

    A.plot(lo_, la_, color=TCOL[925], lw=2.4, transform=PC, zorder=6,
           path_effects=[pe.withStroke(linewidth=4.0, foreground="white")])
    A.plot(INFLOW[1], INFLOW[0], marker="*", ms=16, mfc="k", mec="white", mew=1.2,
           transform=PC, zorder=7)
    A.plot(lo_[-1], la_[-1], marker="o", ms=8, mfc=TCOL[925], mec="white", mew=1.4,
           transform=PC, zorder=7)
    A.set_title("(a)  Soil moisture on 27 July with the 925 hPa path", loc="left",
                fontsize=10, pad=6)

    # ---------------- (b) how the land partitioned its energy ----------------
    B = F.add_axes([0.545, 0.545, 0.40, 0.375], projection=PC)
    B.set_extent([-104, -82, 25, 47], crs=PC)
    B.add_feature(cfeature.OCEAN.with_scale("50m"), fc="#dceaf2", zorder=2)
    B.add_feature(cfeature.STATES.with_scale("50m"), ec="0.55", lw=0.4, zorder=3)
    B.add_feature(cfeature.COASTLINE.with_scale("50m"), ec="0.2", lw=0.7, zorder=3)

    lh27, sh27 = FLX["2026-07-27"]["lhtfl"], FLX["2026-07-27"]["shtfl"]
    ef27 = np.where(onland & (lh27 + sh27 > 50), lh27 / (lh27 + sh27), np.nan)
    cf2 = B.contourf(LO3, LA3, np.clip(ef27, 0, 1), levels=np.arange(0.1, 1.01, 0.05),
                     cmap="YlGnBu", extend="both", transform=PC, zorder=1)
    cax2 = F.add_axes([0.952, 0.545, 0.011, 0.375])
    cb2 = F.colorbar(cf2, cax=cax2)
    cb2.set_label("Evaporative fraction  LH / (LH + SH),  18\u201321 UTC", fontsize=8)
    cb2.ax.tick_params(labelsize=7)

    B.plot(lo_, la_, color=TCOL[925], lw=2.4, transform=PC, zorder=6,
           path_effects=[pe.withStroke(linewidth=4.0, foreground="white")])
    B.plot(INFLOW[1], INFLOW[0], marker="*", ms=16, mfc="k", mec="white", mew=1.2,
           transform=PC, zorder=7)
    for lab, la0, lo0, col in [("dry, heating\n(Bowen 1.9)", 35.5, -98.6, "#8a4a00"),
                               ("wet, evaporating\n(Bowen 0.04)", 41.6, -91.5, "#0d4f6c")]:
        B.text(lo0, la0, lab, fontsize=7.4, fontweight="bold", ha="center", color=col,
               transform=PC, zorder=8,
               path_effects=[pe.withStroke(linewidth=2.2, foreground="white")])
    B.set_title("(b)  How the land partitioned its energy, 27 July afternoon", loc="left",
                fontsize=10, pad=6)

    # ---------------- (c) along-path profile ----------------
    C = F.add_axes([0.055, 0.290, 0.89, 0.185])
    xh = LP.hours_before.values
    C.plot(xh, LP.soilw_0_10cm, color="#7a4a10", lw=2.0, label="0\u201310 cm soil moisture", zorder=4)
    C.fill_between(xh, 0, LP.soilw_0_10cm, color="#7a4a10", alpha=0.12, zorder=2)
    C.set_ylabel("Soil moisture (m\u00b3 m$^{-3}$)", color="#7a4a10")
    C.tick_params(axis="y", labelcolor="#7a4a10")
    C.set_ylim(0, 0.33); C.set_xlim(-96, 2); C.grid(alpha=0.25, zorder=0)

    C2 = C.twinx()
    C2.plot(xh, LP.ef_clip, color="#0d4f6c", lw=2.2, label="evaporative fraction", zorder=5)
    C2.axhline(0.5, color="0.5", ls=":", lw=1.0, zorder=1)
    C2.set_ylabel("Evaporative fraction", color="#0d4f6c")
    C2.tick_params(axis="y", labelcolor="#0d4f6c")
    C2.set_ylim(0, 1.15)

    for (a, b, lab), col in zip(legs, ["#b08040", "#c89a50", "#7fb0c0", "#4a90a8"]):
        C.axvspan(a, b, color=col, alpha=0.13, zorder=0)
        C.text((a + b) / 2, 0.312, lab, ha="center", fontsize=7.4, color="0.25", fontweight="bold")

    C.set_xlabel("Hours before arrival over Chicago")
    C.set_title("(c)  Along the path: the surface flips from heating the air to moistening it",
                loc="left", fontsize=10, pad=6)
    h1, l1 = C.get_legend_handles_labels()
    h2, l2 = C2.get_legend_handles_labels()
    C.legend(h1 + h2, l1 + l2, loc="lower left", fontsize=7.6, framealpha=0.93, ncol=2)

    # ---------------- (d) observed antecedent rainfall ----------------
    D = F.add_axes([0.055, 0.055, 0.40, 0.165])
    pn = PATHN.iloc[::-1]
    y = np.arange(len(pn))
    cols = ["#8c510a" if v < 50 else "#d8b365" if v < 90 else "#5ab4ac" if v < 150 else "#01665e"
            for v in pn.pct_of_normal]
    D.barh(y, pn.pct_of_normal, color=cols, zorder=3, height=0.62)
    D.axvline(100, color="0.3", ls="--", lw=1.2, zorder=4)
    D.set_yticks(y)
    D.set_yticklabels([l.split(" (")[0] for l in pn.label], fontsize=7.6)
    for yi, v, p in zip(y, pn.pct_of_normal, pn.percentile):
        D.text(v + 6, yi, f"{v:.0f}%  ({p:.0f}th pctile)", va="center", fontsize=7.2, color="0.25")
    D.set_xlabel("30-day rainfall as % of 1995\u20132025 normal")
    D.set_xlim(0, 300); D.grid(axis="x", alpha=0.25, zorder=0)
    D.set_title("(d)  Observed antecedent wetness at stations near the path", loc="left",
                fontsize=10, pad=6)

    # ---------------- (e) four-leg moisture budget ----------------
    E = F.add_axes([0.545, 0.055, 0.40, 0.165])
    xx = np.arange(len(AG))
    cols = ["#8c510a" if v < 0.45 else "#d8b365" if v < 0.7 else "#01665e" for v in AG.EF]
    E.bar(xx, AG.dq_per_day, color=cols, zorder=3, width=0.6)
    E.axhline(0, color="0.25", lw=1.1, zorder=4)
    for xi, v, ef, bw in zip(xx, AG.dq_per_day, AG.EF, AG.bowen):
        E.text(xi, v + (0.28 if v >= 0 else -0.28), f"{v:+.1f}", ha="center",
               va="bottom" if v >= 0 else "top", fontsize=8.4, fontweight="bold",
               color="#01665e" if v >= 0 else "#8c510a")
        E.text(xi, -4.55, f"EF {ef:.2f}\nBowen {bw:.2f}", ha="center", va="bottom",
               fontsize=6.8, color="0.3")
    E.set_xticks(xx)
    E.set_xticklabels([l.replace(" / ", "/\n") for l in AG.leg], fontsize=7.4)
    E.set_ylabel("Moisture change\n(g kg$^{-1}$ per day)")
    E.set_ylim(-5.2, 6.2); E.grid(axis="y", alpha=0.25, zorder=0)
    E.set_title("(e)  The parcel lost vapour over dry ground and regained it over wet ground",
                loc="left", fontsize=10, pad=6)
    E.annotate("evaporation from\nthe Gulf + wet TX coast", xy=(0, 4.35), xytext=(0.42, 5.5),
               fontsize=6.9, color="#01665e", ha="center",
               arrowprops=dict(arrowstyle="->", lw=0.8, color="#01665e"))
    E.annotate("dry, hot Plains:\nmixing dilutes the plume", xy=(1.28, -1.9), xytext=(0.60, 0.80),
               textcoords="axes fraction", fontsize=6.8, color="#8c510a", ha="center",
               arrowprops=dict(arrowstyle="->", lw=0.8, color="#8c510a",
                               connectionstyle="arc3,rad=0.2"))

    F.suptitle("Soil conditions along the moisture path \u2014 did wet ground protect the vapour?",
               x=0.055, ha="left", fontsize=13.5, y=0.982)
    F.text(0.055, 0.944,
           "Soil moisture: GFS 0.25\u00b0 analysis (SOILW, 0\u201310 cm), 27 Jul 12 UTC. Surface "
           "fluxes: same cycle, 6\u20139 h average = 18\u201321 UTC (local afternoon).\nGFS soil "
           "moisture is the Noah land-surface model state inside the GFS assimilation, not an "
           "observation \u2014 panel (d) is the independent observational check (Iowa Environmental "
           "Mesonet\nstation rainfall, normalised to the same 30-day window in 1995\u20132025). "
           "Evaporative fraction LH/(LH+SH) is the share of surface energy going into evaporation "
           "rather than warming the air.",
           fontsize=8.4, color=META_GREY, ha="left", va="top")
    return F


figL = build_soil_figure()
figL.savefig("soil_moisture_path.png", dpi=200, bbox_inches="tight")

![Soil conditions along the moisture path: GFS 0-10 cm soil moisture and afternoon evaporative fraction with the 925 hPa trajectory overlaid, the along-path profile, observed antecedent rainfall as a percentage of normal at seven stations, and the four-leg parcel moisture budget](02_moisture_source_and_transport_outputs/soil_moisture_path.png)

## 12. The link to the flooding

This is the point worth carrying into the interview, and it is the hinge between this notebook and
Notebook 3.

**The same wet ground that fed extra moisture into the 27 July storms is the ground that could not
absorb the rain four days later.** Saturated soil across Iowa and Illinois did two things at once:
it drove the evaporative fraction to 0.95, adding the final +3.2 g kg⁻¹ per day to the inflow that
helped make the airmass extraordinarily unstable — and it left almost no storage capacity for
infiltration, so that when the following rainfall arrived, a much larger fraction of it ran off into
the creeks and rivers instead of soaking in.

Wet soil is **simultaneously an ingredient for severe convection and a precondition for flooding.**
These are not two unrelated weather events that happened to occur in the same week. They share a land
surface, and that shared land surface is a causal link, not a coincidence. Notebook 3 takes the
flooding side of it.

## 13. What we learned

1. **The low-level moisture was Gulf-sourced, and only the low levels were.** The 925 hPa parcel
   arriving in the Glen Ellyn inflow was over the Texas coast / northwest Gulf (29.43 °N, 97.20 °W)
   four days earlier. At 850 hPa the air came from the southern High Plains; at 700 hPa from Montana.
2. **The parcel gained 44% more vapour en route**, 12.5 → 18.1 g kg⁻¹. The transport was not merely
   advective — the path added water.
3. **The 925 hPa path is the one that matters** because 43% of the 66.1 mm column precipitable water
   sat below 850 hPa. The comparative dryness above the moist boundary layer is favourable in its own
   right: it steepens lapse rates.
4. **Transport was of atmospheric-river magnitude.** IVT at the inflow point reached
   822 kg m⁻¹ s⁻¹ at 17 UTC, against a ~250 classification threshold and a typical summer Midwest
   value of 200–400.
5. **Observations confirm the arrival.** O'Hare daily max dewpoint rose 48 → 79 °F in four days,
   landing at the 99.7th percentile of every July day since 1990 — 14 days in 1147.
6. **The path was not uniformly wet; there was a dry gap.** The parcel lost 2.0 g kg⁻¹ per day
   crossing the southern Plains (soil moisture 0.106 m³ m⁻³, Bowen 1.89) and regained 3.2 g kg⁻¹ per
   day over Iowa and Illinois (0.195 m³ m⁻³, EF 0.95, Bowen 0.04). **Wet soil at the receiving end
   mattered more than wet soil at the source.**
7. **Independent gauges reproduce the soil pattern**: Wichita at 9% of normal 30-day rainfall
   (3rd percentile) marks the dry gap; Des Moines at 220% (97th) and O'Hare at 138% (84th) mark the
   wet receiving end.
8. **An appealing hourly correlation did not survive testing** — r = +0.10, p = 0.35, n = 97 — and
   was replaced by an aggregated budget whose values are reported as values.

## 14. Caveats and limitations

**The trajectories are isobaric.** Parcels are advected along a constant pressure surface rather than
following true three-dimensional motion, so vertical displacement — which in reality is where much of
the moisture exchange happens — is absent by construction. Over four days of boundary-layer flow
across relatively flat terrain this is a reasonable approximation, and it is more than sufficient to
attribute a *source region*. But the **exact path should not be over-read**: a kinematic trajectory
using the model vertical velocity, or a proper HYSPLIT calculation with its dispersion treatment,
would refine both the path and the level at which the parcel actually travelled.

**Trajectory moisture is sampled from GFS analyses**, so the humidity history in Figure 1(b) reflects
the model's humidity field, not measurements. The O'Hare dewpoint record is the independent check on
the arrival value, and it agrees; there is no comparable independent check on the interior of the
path.

**The leg-by-leg dq values are not a formal moisture budget.** Specific humidity along a trajectory
changes through several processes: surface evaporation, entrainment and mixing with air of different
humidity above and beside the parcel, and removal by condensation and rainfall. We attribute the
along-path pattern to the land surface because the surface energy partition covaries with it in the
expected direction and the independent rainfall data support the same spatial structure — but a
closed budget would need to quantify the mixing and precipitation terms separately, and we have not
done that. The sign changes are robust; the magnitudes should be read as indicative.

**GFS soil moisture is a Noah land-surface model state**, spun up by GFS's own precipitation history
and unconstrained by soil observations. Where GFS mis-forecast rain, its soil is wrong. Panel (d) of
Figure 2 exists for this reason, and the agreement between the two is the argument.

**Surface fluxes are 3-hour averages over 18–21 UTC.** That window is local afternoon over the
Plains, which is the right choice for peak land–atmosphere coupling, but it is *not* local afternoon
everywhere in the domain simultaneously, and a parcel crossing at a different local time saw
different fluxes than the ones we sampled.

**Evaporative fraction is ill-conditioned over very wet surfaces**, where sensible heat can go
slightly negative. We guard the denominator, report the Bowen ratio alongside, and clip only for
display — but an EF of 0.95 over Iowa should be read as "nearly all available energy went to
evaporation", not as a precisely measured 0.95.

Analysis by Scott Collis and Claude.